In [ ]:
""" 
Referans Noktalarımız olan:
- Denver, CO (Yarı Kurak / Referans) Koordinatlar: 39.74, -105.18
- Miami, FL (Nemli / Tropikal / Kaotik) Koordinatlar: 25.76, -80.19
- Phoenix, AZ (Çöl / Çok Pürüzsüz) Koordinatlar: 33.44, -112.07
- Seattle, WA (Okyanusal / Sürekli Kapalı) Koordinatlar: 47.60, -122.33
- Boston, MA (Karasal / Karlı Kış) Koordinatlar: 42.36, -71.05
2020 ve 2024 yılları arasındaki  verileri NREL API'si üzerinden çekmek istiyoruz.
Bu kod parçası, belirtilen yıllar için verileri çekip birleştirecek ve ardından tek bir DataFrame olarak kaydedecektir.
"""

import sys # Python arama yolunu güncellemek için sys modülünü ekleyelim
import os # Dosya yollarını yönetmek için os modülünü ekleyelim
import importlib # Dinamik modül yükleme ve yeniden yükleme için importlib'i ekleyelim
import pandas as pd # Veri işleme için pandas'ı ekleyelim
from dotenv import load_dotenv # .env dosyasındaki değişkenleri yüklemek için python-dotenv kütüphanesinden load_dotenv fonksiyonunu ekleyelim
import pyarrow as pa # Parquet formatında veri kaydetmek için pyarrow kütüphanesini ekleyelim
import pyarrow.parquet as pq # Parquet dosyalarını yönetmek için pyarrow.par


# Proje kök dizinini Python arama yolunda en üste al
project_root = os.path.abspath(os.path.join('..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root) # Proje kök dizinini arama yolunun en üstüne ekleyelim

else:
    sys.path.remove(project_root) # Eğer zaten varsa, önce kaldırıp sonra tekrar ekleyelim ki güncel versiyonun yüklendiğinden emin olalım
    sys.path.insert(0, project_root)

# Yerel modülü yükle ve cache kaynaklı eski sürümü engelle
import src.data_loader as data_loader # data_loader modülünü src klasöründen içe aktaralım
importlib.reload(data_loader) # Modülü yeniden yükleyelim ki yapılan değişiklikler yansısın
fetch_nrel_data = data_loader.fetch_nrel_data # fetch_nrel_data fonksiyonunu data_loader modülünden alalım
print("Kullanılan data_loader:", data_loader.__file__) 


# .env dosyasındaki değişkenleri yükle
load_dotenv()

# API anahtarınızı, konumu ve çekilecek yılları buradan ayarlayın
api_key = os.getenv("NREL_API_KEY") 

# 5 Stratejik İklim Konumu
cities_dict = {
    "Denver_CO": (39.74, -105.18),    # Yarı Kurak / Referans
    "Miami_FL": (25.76, -80.19),      # Nemli / Tropikal / Kaotik
    "Phoenix_AZ": (33.44, -112.07),   # Çöl / Çok Pürüzsüz
    "Seattle_WA": (47.60, -122.33),   # Okyanusal / Sürekli Kapalı
    "Boston_MA": (42.36, -71.05)      # Karasal / Karlı Kış
}
# 5 Yıllık Veri Aralığı
years_list = [2020, 2021, 2022, 2023, 2024]

# SÜPER FONKSİYONU ÇAĞIR
my_df = data_loader.build_multicity_dataset(api_key, cities_dict, years_list)

if my_df is not None:
    # 1. Datetime Indexing İşlemi
    my_df['datetime'] = pd.to_datetime(
        my_df[['Year','Month','Day','Hour','Minute']].rename(columns={
            'Year':'year', 'Month':'month', 'Day':'day', 'Hour': 'hour', 'Minute':'minute'
        })
    )
    my_df.set_index('datetime', inplace=True)
    
    # 2. Memory Optimization (Downcasting)
    for col in my_df.columns:
        if my_df[col].dtype == 'int64':
            my_df[col] = my_df[col].astype('int32')
        elif my_df[col].dtype == 'float64':
            my_df[col] = my_df[col].astype('float32')

    # 3. Parquet Olarak Kaydet (Klasör yapısına uygun)
    save_path = '../data/nrel_data_5cities_2020_2024.parquet'
    table = pa.Table.from_pandas(my_df)
    pq.write_table(table, save_path)
    
    print(f"\n İşlem Tamamlandı! Toplam Satır: {len(my_df):,}")
    print(f"Veri '{save_path}' konumuna kaydedildi.")
    print(my_df['City'].value_counts())
else:
    print("Veri çekilemedi.")

Kullanılan data_loader: /Users/melih/Solar-Forecasting-XAI/src/data_loader.py

 Denver_CO bölgesi için veri hasadı başlıyor...
   -> 2020 verisi çekiliyor...
   -> 2021 verisi çekiliyor...
   -> 2022 verisi çekiliyor...
   -> 2023 verisi çekiliyor...
   -> 2024 verisi çekiliyor...

 Miami_FL bölgesi için veri hasadı başlıyor...
   -> 2020 verisi çekiliyor...
   -> 2021 verisi çekiliyor...
   -> 2022 verisi çekiliyor...
   -> 2023 verisi çekiliyor...
   -> 2024 verisi çekiliyor...

 Phoenix_AZ bölgesi için veri hasadı başlıyor...
   -> 2020 verisi çekiliyor...
   -> 2021 verisi çekiliyor...
   -> 2022 verisi çekiliyor...
   -> 2023 verisi çekiliyor...
   -> 2024 verisi çekiliyor...

 Seattle_WA bölgesi için veri hasadı başlıyor...
   -> 2020 verisi çekiliyor...
   -> 2021 verisi çekiliyor...
   -> 2022 verisi çekiliyor...
   -> 2023 verisi çekiliyor...
   -> 2024 verisi çekiliyor...

 Boston_MA bölgesi için veri hasadı başlıyor...
   -> 2020 verisi çekiliyor...
   -> 2021 verisi çekiliy

In [4]:
# Parquet dosyasından veriyi alıp pandas'a aktarma

table = pq.read_table('../data/nrel_data_5cities_2020_2024.parquet')

df = table.to_pandas()

print(df)

                     Year  Month  Day  Hour  Minute  GHI  DHI  DNI  \
datetime                                                             
2020-01-01 00:00:00  2020      1    1     0       0    0    0    0   
2020-01-01 00:30:00  2020      1    1     0      30    0    0    0   
2020-01-01 01:00:00  2020      1    1     1       0    0    0    0   
2020-01-01 01:30:00  2020      1    1     1      30    0    0    0   
2020-01-01 02:00:00  2020      1    1     2       0    0    0    0   
...                   ...    ...  ...   ...     ...  ...  ...  ...   
2024-12-31 21:30:00  2024     12   31    21      30    0    0    0   
2024-12-31 22:00:00  2024     12   31    22       0    0    0    0   
2024-12-31 22:30:00  2024     12   31    22      30    0    0    0   
2024-12-31 23:00:00  2024     12   31    23       0    0    0    0   
2024-12-31 23:30:00  2024     12   31    23      30    0    0    0   

                     Wind Speed  Temperature  Cloud Type  Dew Point  \
datetime          